### The University of Melbourne, School of Computing and Information Systems
# COMP90086 Computer Vision, 2025 Semester 2

## FINAL PROJECT

In [1]:
import tensorflow as tf
import pandas as pd
import os
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Configuration ---
IMG_HEIGHT = 640
IMG_WIDTH = 480
BATCH_SIZE = 32
BASE_DIR = "Nutrition5k"
VALIDATION_SPLIT = 0.20 # 20% for validation
RANDOM_SEED = 42 # Use a seed for reproducible splits

# --- 1. Load the full labeled dataset manifest ---
full_labels_path = os.path.join(BASE_DIR, "nutrition5k_train.csv")
df = pd.read_csv(full_labels_path)
df['ID'] = df['ID'].astype(str)




In [2]:
# complete train and test split (with normalised pixel values)
df = df.assign(train_image_path=BASE_DIR + "\\train\\color\\" + df["ID"].astype(str) + "\\rgb.png",
               test_image_path=BASE_DIR + "\\test\\color\\" + df["ID"].astype(str) + "\\rgb.png")

datagen = ImageDataGenerator(rescale=1./255,
                             validation_split=VALIDATION_SPLIT)

train_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

Found 2641 validated image filenames.
Found 660 validated image filenames.


In [3]:
os.listdir(os.path.join(BASE_DIR, "test/color"))

['dish_3301',
 'dish_3302',
 'dish_3303',
 'dish_3304',
 'dish_3305',
 'dish_3306',
 'dish_3307',
 'dish_3308',
 'dish_3309',
 'dish_3310',
 'dish_3311',
 'dish_3312',
 'dish_3313',
 'dish_3314',
 'dish_3315',
 'dish_3316',
 'dish_3317',
 'dish_3318',
 'dish_3319',
 'dish_3320',
 'dish_3321',
 'dish_3322',
 'dish_3323',
 'dish_3324',
 'dish_3325',
 'dish_3326',
 'dish_3327',
 'dish_3328',
 'dish_3329',
 'dish_3330',
 'dish_3331',
 'dish_3332',
 'dish_3333',
 'dish_3334',
 'dish_3335',
 'dish_3336',
 'dish_3337',
 'dish_3338',
 'dish_3339',
 'dish_3340',
 'dish_3341',
 'dish_3342',
 'dish_3343',
 'dish_3344',
 'dish_3345',
 'dish_3346',
 'dish_3347',
 'dish_3348',
 'dish_3349',
 'dish_3350',
 'dish_3351',
 'dish_3352',
 'dish_3353',
 'dish_3354',
 'dish_3355',
 'dish_3356',
 'dish_3357',
 'dish_3358',
 'dish_3359',
 'dish_3360',
 'dish_3361',
 'dish_3362',
 'dish_3363',
 'dish_3364',
 'dish_3365',
 'dish_3366',
 'dish_3367',
 'dish_3368',
 'dish_3369',
 'dish_3370',
 'dish_3371',
 'dish

In [4]:
# create dataset to load test dataset
df_test = pd.DataFrame({"ID": os.listdir(os.path.join(BASE_DIR, "test/color"))})
df_test = df_test.assign(test_image_path = BASE_DIR + "\\test\\color\\" + df_test["ID"].astype(str) + "\\rgb.png")

In [5]:
df_test

,ID,test_image_path
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png
...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png


In [6]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator=test_datagen.flow_from_dataframe(
dataframe=df_test,
directory=os.getcwd(),
x_col="test_image_path",
y_col=None,
target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
batch_size=BATCH_SIZE,
seed=RANDOM_SEED,
shuffle=False,
class_mode=None)

Found 189 validated image filenames.


In [7]:
# build basic regression model
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        
        layers.Conv2D(16, (5, 5), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in

        # this commented out code was part of iteration 1
        #layers.Conv2D(2, (5, 5), activation='relu'), # fill in
        #layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])

In [8]:
STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size

model.fit(train_generator,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=validation_generator,
          validation_steps=STEP_SIZE_VALID,
          epochs=20
)


c:\Users\nares\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 66s 795ms/step - loss: 47220.7812 - mse: 47220.7812 - val_loss: 27122.2246 - val_mse: 27122.2246
Epoch 2/20
 1/82 ━━━━━━━━━━━━━━━━━━━━ 37s 466ms/step - loss: 31602.3418 - mse: 31602.3418

c:\Users\nares\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 101ms/step - loss: 31602.3418 - mse: 31602.3418 - val_loss: 25433.0508 - val_mse: 25433.0508
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 63s 772ms/step - loss: 31812.9102 - mse: 31812.9102 - val_loss: 25756.5000 - val_mse: 25756.5000
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 102ms/step - loss: 31317.4141 - mse: 31317.4141 - val_loss: 25746.8906 - val_mse: 25746.8906
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 72s 884ms/step - loss: 28287.3594 - mse: 28287.3594 - val_loss: 23244.5000 - val_mse: 23244.5000
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - loss: 16059.0674 - mse: 16059.0674 - val_loss: 23139.9844 - val_mse: 23139.9844
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 65s 798ms/step - loss: 21183.7715 - mse: 21183.7715 - val_loss: 23446.9062 - val_mse: 23446.9062
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - loss: 25971.8105 - mse: 25971.8105 - val_loss: 22862.2812 - val_mse: 22862.2812
Epoch 9/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 68s 823ms/step - loss: 24457.863

In [9]:
# validation
STEP_SIZE_TEST=test_generator.n//test_generator.batch_size
model.evaluate(validation_generator)

21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 398ms/step - loss: 14291.3438 - mse: 14291.3438


[14954.1484375, 14954.1484375]

In [10]:
test_generator.reset()
preds=model.predict(test_generator)

6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 427ms/step


In [11]:
df_test = df_test.assign(Value=preds)

In [12]:
df_test

,ID,test_image_path,Value
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png,732.163818
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png,170.205246
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png,74.106445
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png,220.635132
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png,401.306580
...,...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png,144.152679
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png,35.887791
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png,308.320618
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png,155.971771


In [13]:
df_submit = df_test.drop("test_image_path", axis=1)

In [14]:
df_submit

,ID,Value
0,dish_3301,732.163818
1,dish_3302,170.205246
2,dish_3303,74.106445
3,dish_3304,220.635132
4,dish_3305,401.306580
...,...,...
184,dish_3485,144.152679
185,dish_3486,35.887791
186,dish_3487,308.320618
187,dish_3488,155.971771


In [ ]:
# save copy to csv
df_submit.to_csv("interation_2_submission.csv", index=False)